# 지식 증류 (Knowledge Distillation)
Teacher Model(선생님 모델)
- 모델의 크기가 크다
- 똑똑하지만 느리고 무겁다
- 정확도가 높다

Student Model
- 모델 크기가 작다
- 가볍고 빠르다
- 효율적이다


일반학습과의 차이점 : 단순이 1과 2를 가르치는 것이 아닌, 1,2외의 제3을 언급하여 대상을 가르친다

    ex ) 고양이와 강아지랑 좀 비슷하고, 새와는 완전히 달라

In [1]:
import numpy as np
import tensorflow as tf
from tqdm.notebook import tqdm

In [2]:
t_epoch = 10
s_epoch = 5
learning_rate = 0.01
batch_size = 64
temperature = 3
alpha = 0.5

In [4]:
# 데이터셋 불러오기
(X_train,y_train),(X_test,y_test) = tf.keras.datasets.mnist.load_data()

X_train = X_train.astype('float32')/255.0
X_train = np.reshape(X_train,(-1,28,28,1))

X_test = X_test.astype('float32')/255.0
X_test = np.reshape(X_test,(-1,28,28,1))

In [8]:
# Teacher 모델
i = tf.keras.Input(shape=(28,28,1))
out = tf.keras.layers.Conv2D(256,(3,3),strides=(2,2),padding='same')(i)
out = tf.keras.layers.LeakyReLU(alpha=0.2)(out)
out = tf.keras.layers.MaxPooling2D(pool_size=(2,2),strides=(2,2),padding='same')(out)
out = tf.keras.layers.Conv2D(512,(3,3),strides=(2,2),padding='same')(out)
out = tf.keras.layers.Flatten()(out)
out = tf.keras.layers.Dense(10)(out)
t_model = tf.keras.Model(inputs=[i],outputs=[out])

t_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_2 (Conv2D)           (None, 14, 14, 256)       2560      
                                                                 
 leaky_re_lu_1 (LeakyReLU)   (None, 14, 14, 256)       0         
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 7, 7, 256)        0         
 2D)                                                             
                                                                 
 conv2d_3 (Conv2D)           (None, 4, 4, 512)         1180160   
                                                                 
 flatten_1 (Flatten)         (None, 8192)              0         
                                                             

In [11]:
i = tf.keras.Input(shape=(28,28,1))
out = tf.keras.layers.Flatten()(i)
out = tf.keras.layers.Dense(28)(out)
out = tf.keras.layers.Dense(10)(out)

s_model_01 = tf.keras.Model(inputs=i, outputs=out)
s_model_02 = tf.keras.models.clone_model(s_model_01)

s_model_01.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 flatten_4 (Flatten)         (None, 784)               0         
                                                                 
 dense_5 (Dense)             (None, 28)                21980     
                                                                 
 dense_6 (Dense)             (None, 10)                290       
                                                                 
Total params: 22,270
Trainable params: 22,270
Non-trainable params: 0
_________________________________________________________________


In [13]:
t_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

s_model_01.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

s_model_02.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

In [14]:
# Teacher 모델

t_model.fit(X_train,y_train,batch_size=batch_size,epochs=t_epoch)

Epoch 1/10
938/938 [==============================] - 8s 4ms/step - loss: 2.4112 - sparse_categorical_accuracy: 0.9008
Epoch 2/10
938/938 [==============================] - 3s 3ms/step - loss: 1.5814 - sparse_categorical_accuracy: 0.9489
Epoch 3/10
938/938 [==============================] - 3s 3ms/step - loss: 1.6944 - sparse_categorical_accuracy: 0.9496
Epoch 4/10
938/938 [==============================] - 3s 3ms/step - loss: 3.3995 - sparse_categorical_accuracy: 0.9478
Epoch 5/10
938/938 [==============================] - 3s 3ms/step - loss: 2.3937 - sparse_categorical_accuracy: 0.9602
Epoch 6/10
938/938 [==============================] - 3s 4ms/step - loss: 2.9484 - sparse_categorical_accuracy: 0.9604
Epoch 7/10
938/938 [==============================] - 3s 4ms/step - loss: 3.0118 - sparse_categorical_accuracy: 0.9617
Epoch 8/10
938/938 [==============================] - 3s 3ms/step - loss: 3.0201 - sparse_categorical_accuracy: 0.9634
Epoch 9/10
938/938 [============================

In [15]:
# Student 손실 함수
s_loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# distillation 손실 함수
d_loss = tf.keras.losses.KLDivergence()

In [18]:
batch_count = X_train.shape[0]//batch_size

opt = tf.keras.optimizers.Adam(learning_rate)

for e in range(s_epoch):
    for _ in range(batch_count):
        batch_num = np.random.randint(0,X_train.shape[0],size=batch_size)
        t_pred = t_model.predict(X_train[batch_num])

        with tf.GradientTape() as tape:
            s_pred_01 = s_model_01(X_train[batch_num])
            student_loss = s_loss(y_train[batch_num],s_pred_01)
            distillation_loss = d_loss(tf.nn.softmax(t_pred/temperature,axis=1),
                                      tf.nn.softmax(s_pred_01/temperature,axis=1))
            loss = alpha*student_loss + (1-alpha)*distillation_loss

        vars = s_model_01.trainable_variables
        grad = tape.gradient(loss,vars)
        opt.apply_gradients(zip(grad,vars))

    print('에포크 {}'.format(e))
    print('선생님에게 배운 경우')
    s_model_01.evaluate(X_test,y_test)

    print('혼자 공부한 경우')
    s_model_02.evaluate(X_test,y_test)
    print('\n')

2/2 [==============================] - 0s 22ms/step
에포크 0
선생님에게 배운 경우
313/313 [==============================] - 1s 2ms/step - loss: 0.4336 - sparse_categorical_accuracy: 0.9075
혼자 공부한 경우
313/313 [==============================] - 1s 2ms/step - loss: 2.5666 - sparse_categorical_accuracy: 0.1378


2/2 [==============================] - 0s 14ms/step
에포크 1
선생님에게 배운 경우
313/313 [==============================] - 1s 2ms/step - loss: 0.4509 - sparse_categorical_accuracy: 0.9088
혼자 공부한 경우
313/313 [==============================] - 1s 2ms/step - loss: 2.5666 - sparse_categorical_accuracy: 0.1378


2/2 [==============================] - 0s 4ms/step
에포크 2
선생님에게 배운 경우
313/313 [==============================] - 1s 2ms/step - loss: 0.4291 - sparse_categorical_accuracy: 0.9133
혼자 공부한 경우
313/313 [==============================] - 1s 2ms/step - loss: 2.5666 - sparse_categorical_accuracy: 0.1378


2/2 [==============================] - 0s 6ms/step
에포크 3
선생님에게 배운 경우
313/313 [=============================